In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer

dataset = load_dataset("imdb")


model_name =  "prajjwal1/bert-tiny"


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    use_safetensors=True
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )


tokenized = dataset.map(tokenize, batched=True)

# Remove text column and set torch format
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")


collator = DataCollatorWithPadding(tokenizer=tokenizer)








Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
training_args = TrainingArguments(
    output_dir="./sentiment-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=4,   # CPU-friendly
    per_device_eval_batch_size=4,
    num_train_epochs=1,              # Keep it light for testing
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
)

In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"].shuffle(seed=42).select(range(5000)),  # sample for speed
    eval_dataset=tokenized["test"].select(range(2000)),
    data_collator=collator,
    tokenizer=tokenizer
)


trainer.train()



C:\Users\aswin\AppData\Local\Temp\ipykernel_21404\2652429214.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,0.623400,0.572195


TrainOutput(global_step=1250, training_loss=0.6575239410400391, metrics={'train_runtime': 476.2688, 'train_samples_per_second': 10.498, 'train_steps_per_second': 2.625, 'total_flos': 6352435200000.0, 'train_loss': 0.6575239410400391, 'epoch': 1.0})

In [44]:

def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    print(probs.argmax().item())
    return "positive" if probs.argmax().item() == 1 else "negative"

print("Prediction:", predict("I  hate it"))

0
Prediction: negative


In [45]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "This movie was amazing!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)
probs = torch.softmax(outputs.logits, dim=1)

print("Positive" if probs.argmax().item() == 1 else "Negative")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

c:\Users\aswin\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aswin\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Positive


In [1]:
text = "It was good, but I honestly don’t understand why people are calling it a flop movie i enjoyed it"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

outputs = model(**inputs)
probs = torch.softmax(outputs.logits, dim=1)

print("Positive" if probs.argmax().item() == 1 else "Negative")

NameError: name 'tokenizer' is not defined